# Tool Drift Pilot\n\nColab orchestration notebook for the NGEN-AI 2026 tool-drift pilot.

In [ ]:
from pathlib import Path\nimport json\nimport os\n\nROOT = Path('/content/tool-drift')\nprint(ROOT)

In [ ]:
# Install dependencies after cloning or copying the repo into /content/tool-drift.\n!pip install -e /content/tool-drift

## Running Modes\n\n- The notebook now defaults to `demo_mode = False` and uses the real pilot subset files.\n- Set `demo_mode = True` only if you want a quick scaffold check.\n- Set `refresh_subsets = True` only if you want to regenerate the pilot subset files from the exporter scripts.\n- Refreshing subsets expects the official benchmark assets to already exist under `external/`.

In [ ]:
import subprocess\nimport sys\nsys.path.insert(0, str(ROOT))\n\nfrom scripts.common import load_yaml, dump_json\nfrom scripts.run_pilot_bfcl import run as run_bfcl\nfrom scripts.run_pilot_dice import run as run_dice

In [ ]:
bfcl_config = load_yaml(ROOT / 'configs/pilot_bfcl.yaml')\ndice_config = load_yaml(ROOT / 'configs/pilot_dice.yaml')\n\n# Default to the real pilot subsets. Flip this to True only for a quick scaffold check.\ndemo_mode = False\nrefresh_subsets = False\n\nbfcl_subset_path = ROOT / 'data' / 'bfcl_pilot_subset.json'\ndice_subset_path = ROOT / 'data' / 'dice_pilot_subset.json'\n\nif refresh_subsets:\n    subprocess.run([\n        sys.executable,\n        str(ROOT / 'scripts/export_bfcl_subset.py'),\n        '--per-category', '5',\n        '--output', str(bfcl_subset_path),\n    ], check=True)\n    subprocess.run([\n        sys.executable,\n        str(ROOT / 'scripts/export_dice_subset.py'),\n        '--count', '20',\n        '--output', str(dice_subset_path),\n    ], check=True)\n\nbfcl_config['data']['bfcl_subset_path'] = str(bfcl_subset_path)\ndice_config['data']['dice_subset_path'] = str(dice_subset_path)\n\nif not demo_mode:\n    assert bfcl_subset_path.exists(), f'Missing BFCL subset: {bfcl_subset_path}'\n    assert dice_subset_path.exists(), f'Missing DICE subset: {dice_subset_path}'\n\n# Edit values here if you want a different output location or sample count.\nbfcl_config['evaluation']['sample_count'] = 8 if demo_mode else 20\ndice_config['evaluation']['sample_count'] = 8 if demo_mode else 20\nbfcl_config['pilot']['demo_mode'] = demo_mode\ndice_config['pilot']['demo_mode'] = demo_mode\n\ndump_json(ROOT / 'configs/pilot_bfcl.runtime.json', bfcl_config)\ndump_json(ROOT / 'configs/pilot_dice.runtime.json', dice_config)\nprint({\n    'demo_mode': demo_mode,\n    'refresh_subsets': refresh_subsets,\n    'bfcl_subset_path': str(bfcl_subset_path),\n    'dice_subset_path': str(dice_subset_path),\n})

In [ ]:
bfcl_result = run_bfcl(bfcl_config, demo=demo_mode)\ndice_result = run_dice(dice_config, demo=demo_mode)\nprint(bfcl_result['summary'])\nprint(dice_result['summary'])

In [ ]:
from scripts.summarize_results import summarize_file\n\noutputs = ROOT / 'outputs'\nall_summaries = [summarize_file(path) for path in sorted(outputs.rglob('*_results.json'))]\nfor row in all_summaries:\n    print(row)

In [ ]:
latest_by_benchmark = {}\nfor row in all_summaries:\n    if not row.get('run_id'):\n        continue\n    latest_by_benchmark[row['benchmark']] = row\n\ntable = []\nfor benchmark, row in sorted(latest_by_benchmark.items()):\n    table.append({\n        'benchmark': benchmark,\n        'run_id': row.get('run_id'),\n        'demo_mode': row.get('demo_mode'),\n        'sample_count': row.get('sample_count'),\n        'original_score': row.get('original_score'),\n        'drifted_score': row.get('drifted_score'),\n        'repaired_score': row.get('repaired_score'),\n        'recovery_rate': row.get('recovery_rate'),\n        'error_breakdown': row.get('error_breakdown'),\n    })\n\ntry:\n    import pandas as pd\n    display(pd.DataFrame(table))\nexcept Exception:\n    for row in table:\n        print(row)